# ☁️ Wind Turbine RAG Project — Azure Deployment Guide

| Phase | Description |
|---|---|
| **Phase 1** | Azure Infrastructure Creation |
| **Phase 2** | AI Services Setup |
| **Phase 3** | CI/CD Pipeline |
| **Phase 4** | Application Deployment |
| **Phase 5** | Monitoring & Observability |

> ⚠️ **Prerequisites:** Signed in to [portal.azure.com](https://portal.azure.com), Docker installed locally, GitHub repo ready.

---
# Phase 1 — Azure Infrastructure Creation
> **Rule:** Always create Key Vault first so every other service can store its secrets there immediately.

## Step 1.1 — Set Environment Variables (CLI only)
Run once — all CLI commands below reference these variables.

In [ ]:
RESOURCE_GROUP='rg-windturbine-prod'
LOCATION='eastus'
KEY_VAULT='kv-windturbine'
STORAGE_ACCOUNT='sawindturbine'
ACR_NAME='acrwindturbine'
CONTAINER_ENV='cae-windturbine'
BACKEND_APP='ca-windturbine-backend'
FRONTEND_APP='ca-windturbine-frontend'
AOAI_NAME='aoai-windturbine'
SEARCH_NAME='srch-windturbine'
COSMOS_NAME='cosmos-windturbine'
REDIS_NAME='redis-windturbine'
FUNCTION_APP='func-windturbine-ingest'
APP_INSIGHTS='appi-windturbine'
LOG_ANALYTICS='law-windturbine'

echo 'Variables set'

## Step 1.2 — Create Resource Group
All project resources live in one group for unified billing and cleanup.

### 🖥️ Azure Portal
1. Go to [portal.azure.com](https://portal.azure.com)
2. Search **'Resource groups'** in the top search bar → click **+ Create**
3. Fill in:
   - **Subscription:** your subscription
   - **Resource group:** `rg-windturbine-prod`
   - **Region:** `East US`
4. Click **Review + create** → **Create**

## Step 1.3 — Create Azure Key Vault
**Create this first.** Stores all API keys and connection strings. Never hardcode credentials anywhere.

### 🖥️ Azure Portal
1. Search **'Key vaults'** → **+ Create**
2. Fill in:
   - **Resource group:** `rg-windturbine-prod`
   - **Key vault name:** `kv-windturbine`
   - **Region:** `East US`
   - **Pricing tier:** Standard
3. Go to **Access configuration** tab → select **Azure role-based access control (RBAC)**
4. Click **Review + create** → **Create**
5. After creation: go to the Key Vault → **Access control (IAM)** → **+ Add role assignment**
   - Role: `Key Vault Administrator` → assign to yourself

## Step 1.4 — Create Storage Account + Blob Containers
Stores raw PDFs/manuals (`raw-documents`) and extracted diagram images (`extracted-images`).

### 🖥️ Azure Portal
1. Search **'Storage accounts'** → **+ Create**
2. Fill in:
   - **Resource group:** `rg-windturbine-prod`
   - **Storage account name:** `sawindturbine`
   - **Region:** `East US`
   - **Redundancy:** `Locally-redundant storage (LRS)`
3. Click **Review + create** → **Create**
4. After creation: go to the Storage Account → **Containers** (left menu)
5. Click **+ Container** → Name: `raw-documents` → Access: **Private** → **Create**
6. Repeat: **+ Container** → Name: `extracted-images` → **Create**
7. Go to **Security + networking → Access keys** → copy **Connection string** → store in Key Vault:
   - Key Vault → **Secrets** → **+ Generate/Import** → Name: `STORAGE-CONNECTION-STRING` → paste value

## Step 1.5 — Create Azure Container Registry (ACR)
Private Docker registry. CI/CD pushes images here; Container Apps pulls from here.

### 🖥️ Azure Portal
1. Search **'Container registries'** → **+ Create**
2. Fill in:
   - **Resource group:** `rg-windturbine-prod`
   - **Registry name:** `acrwindturbine`
   - **Location:** `East US`
   - **Pricing plan:** `Standard`
3. Click **Review + create** → **Create**
4. After creation: go to ACR → **Settings → Access keys** → enable **Admin user**
   - Copy **Login server**, **Username**, **Password** — you'll need these for GitHub Actions secrets

## Step 1.6 — Create Azure Container Apps Environment
Shared runtime for backend and frontend containers — handles networking, TLS, and scaling.

### 🖥️ Azure Portal
1. Search **'Container Apps Environments'** → **+ Create**
2. Fill in:
   - **Resource group:** `rg-windturbine-prod`
   - **Environment name:** `cae-windturbine`
   - **Region:** `East US`
3. Go to **Monitoring** tab → select your Log Analytics workspace (create one if needed)
4. Click **Review + create** → **Create**

## Step 1.7 — Azure Active Directory App Registration + RBAC
Sets up OAuth 2.0 single-tenant authentication and RBAC for engineers, admins, and supervisors.

### 🖥️ Azure Portal
1. Search **'Microsoft Entra ID'** (formerly Azure AD) → **App registrations** → **+ New registration**
2. Fill in:
   - **Name:** `WindTurbineRAG`
   - **Supported account types:** `Accounts in this organizational directory only (Single tenant)`
   - **Redirect URI:** `https://<your-backend-url>/auth/callback`
3. Click **Register**
4. After registration: go to **Certificates & secrets** → **+ New client secret** → copy value
   - Store in Key Vault: Secret name `AAD-CLIENT-SECRET`
5. Go to **API permissions** → **+ Add a permission** → Microsoft Graph → `User.Read`

**RBAC Role Assignments (Portal):**
6. Go to your Resource Group → **Access control (IAM)** → **+ Add role assignment**
   - Role: `Reader` → assign to field engineer users
   - Role: `Contributor` → assign to admins
   - Role: `Reader` → assign to supervisors

---
# Phase 2 — AI Services Setup
> Deploy all Azure AI services. Store every API key in Key Vault immediately after creation.

## Step 2.1 — Deploy Azure OpenAI Service
Two model deployments under one resource: `gpt-4o` for generation, `text-embedding-ada-002` for 1536-dim embeddings.

### 🖥️ Azure Portal
1. Search **'Azure OpenAI'** → **+ Create**
2. Fill in:
   - **Resource group:** `rg-windturbine-prod`
   - **Region:** `East US` (check model availability first at [aka.ms/oai/docs](https://aka.ms/oai/docs))
   - **Name:** `aoai-windturbine`
   - **Pricing tier:** `Standard S0`
3. Click **Review + create** → **Create**

**Deploy models (Azure OpenAI Studio):**
4. After creation → click **Go to Azure OpenAI Studio**
5. Left menu: **Deployments** → **+ Create new deployment**
   - Model: `gpt-4o` → Deployment name: `gpt-4o` → Tokens/min: `10K` → **Deploy**
6. **+ Create new deployment** again:
   - Model: `text-embedding-ada-002` → Deployment name: `text-embedding-ada-002` → **Deploy**
7. Back in Portal: Azure OpenAI resource → **Keys and Endpoint** → copy **KEY 1** and **Endpoint**
   - Store in Key Vault: `AOAI-API-KEY` and `AOAI-ENDPOINT`

## Step 2.2 — Create Azure AI Search Service
Vector + keyword hybrid search index. Supports HNSW algorithm and semantic re-ranking.

### 🖥️ Azure Portal
1. Search **'AI Search'** → **+ Create**
2. Fill in:
   - **Resource group:** `rg-windturbine-prod`
   - **Service name:** `srch-windturbine`
   - **Location:** `East US`
   - **Pricing tier:** `Standard` (required for semantic ranker)
3. Click **Review + create** → **Create**
4. After creation: go to the Search service → **Settings → Keys** → copy **Primary admin key**
   - Store in Key Vault: `SEARCH-API-KEY`
5. Go to **Semantic ranker** tab → enable **Free** tier semantic ranker

**Create the index (Portal):**
6. Left menu: **Search management → Indexes** → **+ Add index** (or use the Python SDK cell below — recommended for vector field config)

### Create the Search Index via Python SDK
Portal index creation does not support vector field configuration — use Python SDK instead.

In [ ]:
# pip install azure-search-documents
import os
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchField,
    SearchFieldDataType, VectorSearch, HnswAlgorithmConfiguration,
    VectorSearchProfile, SemanticConfiguration, SemanticPrioritizedFields,
    SemanticField, SemanticSearch
)
from azure.core.credentials import AzureKeyCredential

endpoint = os.environ['AZURE_SEARCH_ENDPOINT']
key      = os.environ['AZURE_SEARCH_KEY']
client   = SearchIndexClient(endpoint=endpoint, credential=AzureKeyCredential(key))

fields = [
    SimpleField(name='Id',                  type=SearchFieldDataType.String, key=True),
    SearchableField(name='Procedure_Name',  type=SearchFieldDataType.String),
    SearchableField(name='Procedure_Chunk', type=SearchFieldDataType.String),
    SearchableField(name='Procedure_Steps', type=SearchFieldDataType.String),
    SearchableField(name='Alarm',           type=SearchFieldDataType.String),
    SimpleField(name='component_type',      type=SearchFieldDataType.String, filterable=True),
    SimpleField(name='document_version',    type=SearchFieldDataType.String, filterable=True),
    SimpleField(name='version_updated_at',  type=SearchFieldDataType.DateTimeOffset, sortable=True),
    SimpleField(name='image_urls',          type=SearchFieldDataType.Collection(SearchFieldDataType.String)),
    SearchField(
        name='embedding',
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=1536,       # text-embedding-ada-002
        vector_search_profile_name='hnsw-profile'
    ),
]

vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name='hnsw-algo', parameters={'m': 4, 'efConstruction': 400})],
    profiles=[VectorSearchProfile(name='hnsw-profile', algorithm_configuration_name='hnsw-algo')]
)

semantic_config = SemanticConfiguration(
    name='semantic-config',
    prioritized_fields=SemanticPrioritizedFields(
        content_fields=[SemanticField(field_name='Procedure_Chunk')],
        keywords_fields=[SemanticField(field_name='Procedure_Name'), SemanticField(field_name='Alarm')]
    )
)

index = SearchIndex(
    name='windturbine-index',
    fields=fields,
    vector_search=vector_search,
    semantic_search=SemanticSearch(configurations=[semantic_config])
)
client.create_or_update_index(index)
print('Index created: windturbine-index')

## Step 2.3 — Create Azure Cosmos DB
Stores conversation session history for multi-turn context.
> ⚠️ **Critical:** Add composite index on `session_id + timestamp` — missing this caused 1.8–2.2s query latency in production.

### 🖥️ Azure Portal
1. Search **'Azure Cosmos DB'** → **+ Create** → select **Azure Cosmos DB for NoSQL** → **Create**
2. Fill in:
   - **Resource group:** `rg-windturbine-prod`
   - **Account name:** `cosmos-windturbine`
   - **Location:** `East US`
   - **Capacity mode:** `Provisioned throughput`
3. Click **Review + create** → **Create**
4. After creation: **Data Explorer** → **New Database**
   - Database id: `windturbine-db` → OK
5. Select the new database → **New Container**
   - Container id: `sessions`
   - Partition key: `/session_id`
   - Throughput: `400 RU/s` → OK
6. Go to **Settings → Keys** → copy **PRIMARY CONNECTION STRING**
   - Store in Key Vault: `COSMOS-CONNECTION-STRING`

**Add Composite Index (Portal):**
7. Data Explorer → `sessions` container → **Scale & Settings**
8. In the **Indexing Policy** JSON, add under `compositeIndexes`:
```json
"compositeIndexes": [
  [
    { "path": "/session_id", "order": "ascending" },
    { "path": "/timestamp",  "order": "descending" }
  ]
]
```
9. Click **Save**

### Add composite index via Python SDK (recommended)

In [ ]:
# pip install azure-cosmos
from azure.cosmos import CosmosClient
import os

client    = CosmosClient.from_connection_string(os.environ['COSMOS_CONNECTION_STRING'])
db        = client.get_database_client('windturbine-db')
container = db.get_container_client('sessions')

props = container.read()
props['indexingPolicy']['compositeIndexes'] = [
    [
        {'path': '/session_id', 'order': 'ascending'},
        {'path': '/timestamp',  'order': 'descending'}
    ]
]
container.replace_container(indexing_policy=props['indexingPolicy'])
print('Composite index applied — Cosmos DB queries will now be fast')

## Step 2.4 — Create Azure Document Intelligence
Extracts text, tables, and images from PDFs and scanned engineering manuals during ingestion.

### 🖥️ Azure Portal
1. Search **'Document intelligence'** → **+ Create**
2. Fill in:
   - **Resource group:** `rg-windturbine-prod`
   - **Region:** `East US`
   - **Name:** `docint-windturbine`
   - **Pricing tier:** `Standard S0`
3. Click **Review + create** → **Create**
4. After creation: go to resource → **Keys and Endpoint**
   - Copy **KEY 1** → store in Key Vault: `DOCINT-API-KEY`
   - Copy **Endpoint** → store in Key Vault: `DOCINT-ENDPOINT`

## Step 2.5 — Create Azure Cache for Redis
Semantic cache for repeated queries and token-bucket rate-limiting. Reduced OpenAI API calls by ~34% in production.

### 🖥️ Azure Portal
1. Search **'Azure Cache for Redis'** → **+ Create**
2. Fill in:
   - **Resource group:** `rg-windturbine-prod`
   - **DNS name:** `redis-windturbine`
   - **Location:** `East US`
   - **Cache type:** `Basic C1` (1 GB — sufficient for semantic cache)
3. Click **Review + create** → **Create** *(takes ~10 minutes)*
4. After creation: go to resource → **Settings → Access keys**
   - Copy **Primary connection string** → store in Key Vault: `REDIS-CONNECTION-STRING`

---
# Phase 3 — CI/CD Pipeline
> GitHub Actions: push to main → lint/test → Docker build → push to ACR → deploy to Container Apps.

```
GitHub push → CI (lint/test) → Docker build → ACR push → CD (Container Apps update)
```

## Step 3.1 — Dockerfile: FastAPI Backend

In [ ]:
%%writefile Dockerfile.backend
FROM python:3.11-slim AS builder
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir --prefix=/install -r requirements.txt

FROM python:3.11-slim
WORKDIR /app
COPY --from=builder /install /usr/local
COPY ./app ./app
EXPOSE 8000
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "4"]

## Step 3.2 — Dockerfile: React Frontend

In [ ]:
%%writefile Dockerfile.frontend
FROM node:20-alpine AS builder
WORKDIR /app
COPY package*.json .
RUN npm ci
COPY . .
RUN npm run build

FROM nginx:alpine
COPY --from=builder /app/build /usr/share/nginx/html
COPY nginx.conf /etc/nginx/conf.d/default.conf
EXPOSE 3000
CMD ["nginx", "-g", "daemon off;"]

## Step 3.3 — GitHub Actions Workflow
Save as `.github/workflows/deploy.yml`.

### 🖥️ GitHub Portal setup (before running the workflow)
1. Go to your GitHub repo → **Settings → Secrets and variables → Actions**
2. Click **New repository secret** and add:
   - `AZURE_CREDENTIALS` — JSON output from Step 3.4
   - `ACR_USERNAME` — ACR admin username (from ACR → Access keys)
   - `ACR_PASSWORD` — ACR admin password

In [ ]:
%%writefile .github/workflows/deploy.yml
name: Deploy Wind Turbine RAG

on:
  push:
    branches: [main]

env:
  ACR_NAME: acrwindturbine
  RESOURCE_GROUP: rg-windturbine-prod
  BACKEND_APP: ca-windturbine-backend
  FRONTEND_APP: ca-windturbine-frontend

jobs:
  ci:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: '3.11' }
      - run: pip install -r requirements.txt
      - run: pytest tests/ --cov=app --cov-report=term-missing

  build-push:
    needs: ci
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: azure/login@v1
        with:
          creds: ${{ secrets.AZURE_CREDENTIALS }}
      - uses: azure/docker-login@v1
        with:
          login-server: acrwindturbine.azurecr.io
      - name: Build & push backend
        run: |
          docker build -f Dockerfile.backend \
            -t acrwindturbine.azurecr.io/backend:${{ github.sha }} \
            -t acrwindturbine.azurecr.io/backend:latest .
          docker push acrwindturbine.azurecr.io/backend:${{ github.sha }}
          docker push acrwindturbine.azurecr.io/backend:latest
      - name: Build & push frontend
        run: |
          docker build -f Dockerfile.frontend \
            -t acrwindturbine.azurecr.io/frontend:${{ github.sha }} \
            -t acrwindturbine.azurecr.io/frontend:latest .
          docker push acrwindturbine.azurecr.io/frontend:${{ github.sha }}
          docker push acrwindturbine.azurecr.io/frontend:latest

  deploy:
    needs: build-push
    runs-on: ubuntu-latest
    steps:
      - uses: azure/login@v1
        with:
          creds: ${{ secrets.AZURE_CREDENTIALS }}
      - name: Deploy backend
        run: |
          az containerapp update \
            --name ${{ env.BACKEND_APP }} \
            --resource-group ${{ env.RESOURCE_GROUP }} \
            --image acrwindturbine.azurecr.io/backend:${{ github.sha }}
      - name: Deploy frontend
        run: |
          az containerapp update \
            --name ${{ env.FRONTEND_APP }} \
            --resource-group ${{ env.RESOURCE_GROUP }} \
            --image acrwindturbine.azurecr.io/frontend:${{ github.sha }}

## Step 3.4 — Create GitHub Actions Service Principal

In [ ]:
# ⌨️ Azure CLI — creates service principal for GitHub Actions
SUBSCRIPTION_ID=$(az account show --query id -o tsv)

AZURE_CREDENTIALS=$(az ad sp create-for-rbac \
  --name 'sp-windturbine-github' \
  --role Contributor \
  --scopes /subscriptions/$SUBSCRIPTION_ID/resourceGroups/$RESOURCE_GROUP \
  --sdk-auth)

echo 'Copy the JSON below → GitHub repo → Settings → Secrets → AZURE_CREDENTIALS'
echo $AZURE_CREDENTIALS

---
# Phase 4 — Application Deployment
> Deploy FastAPI backend, React frontend, and the event-driven ingestion Azure Function.

## Step 4.1 — Deploy FastAPI Backend Container App
Autoscaling: min 1 replica, max 10. New replica triggered when concurrent requests exceed 20.

### 🖥️ Azure Portal
1. Search **'Container Apps'** → **+ Create**
2. **Basics** tab:
   - **Resource group:** `rg-windturbine-prod`
   - **Container app name:** `ca-windturbine-backend`
   - **Container Apps Environment:** `cae-windturbine`
3. **Container** tab:
   - Uncheck *Use quickstart image*
   - **Image source:** Azure Container Registry
   - **Registry:** `acrwindturbine`
   - **Image:** `backend` | **Tag:** `latest`
   - **CPU:** `1.0` | **Memory:** `2 Gi`
4. **Ingress** tab:
   - Enable ingress → **Accepting traffic from anywhere**
   - **Target port:** `8000`
5. **Scale** tab:
   - Min replicas: `1` | Max replicas: `10`
   - Add scale rule: **HTTP scaling** → Concurrent requests: `20`
6. Click **Review + create** → **Create**

## Step 4.2 — Enable Managed Identity + Inject Secrets from Key Vault
Zero hardcoded secrets — the Container App identity fetches them from Key Vault at runtime.

### 🖥️ Azure Portal
1. Go to `ca-windturbine-backend` → **Settings → Identity**
2. **System assigned** tab → toggle **Status: On** → **Save**
3. Copy the **Object (principal) ID** shown
4. Go to `kv-windturbine` → **Access control (IAM)** → **+ Add role assignment**
   - Role: `Key Vault Secrets User`
   - Assign access to: `Managed identity`
   - Select the backend Container App identity → **Save**
5. Back in Container App → **Settings → Secrets** → **+ Add**:
   - For each secret, source: **Key Vault reference**
   - Reference: `https://kv-windturbine.vault.azure.net/secrets/AOAI-API-KEY`
   - Repeat for: `SEARCH-API-KEY`, `COSMOS-CONNECTION-STRING`, `REDIS-CONNECTION-STRING`, `STORAGE-CONNECTION-STRING`
6. Go to **Containers** tab → **Edit** → add **Environment variables** mapping each secret

## Step 4.3 — Deploy React Frontend Container App

### 🖥️ Azure Portal
1. Search **'Container Apps'** → **+ Create**
2. Same steps as backend but:
   - **Name:** `ca-windturbine-frontend`
   - **Image:** `frontend:latest`
   - **Target port:** `3000`
   - **Environment variable:** `REACT_APP_API_URL` = `https://<backend-fqdn>`
     (find backend FQDN: Container App → Overview → Application URL)
3. Click **Review + create** → **Create**

## Step 4.4 — Deploy Event-Driven Ingestion Pipeline (Azure Function)
Blob trigger: new PDF uploaded → Document Intelligence → chunk → embed → index into AI Search.
Includes the **document versioning fix**: old chunks deleted before new ones are inserted.

### 🖥️ Azure Portal
1. Search **'Function App'** → **+ Create**
2. Fill in:
   - **Resource group:** `rg-windturbine-prod`
   - **Function App name:** `func-windturbine-ingest`
   - **Runtime stack:** `Python` | **Version:** `3.11`
   - **Region:** `East US`
   - **Hosting:** `Consumption (Serverless)`
3. **Storage** tab: select `sawindturbine`
4. Click **Review + create** → **Create**
5. After creation: go to Function App → **Configuration → Application settings** → **+ New setting**
   - Add all connection strings (pull from Key Vault)
6. Deploy function code via VS Code Azure extension or `func azure functionapp publish func-windturbine-ingest`

### Ingestion Function Code — `function_app.py`

In [ ]:
# function_app.py
import azure.functions as func
import logging, os, hashlib
from azure.ai.formrecognizer import DocumentAnalysisClient
from azure.search.documents import SearchClient
from openai import AzureOpenAI
from datetime import datetime, timezone

app = func.FunctionApp()

@app.blob_trigger(
    arg_name='blob',
    path='raw-documents/{name}',
    connection='STORAGE_CONNECTION_STRING'
)
def ingest_document(blob: func.InputStream):
    logging.info(f'Processing: {blob.name}')

    # 1. Extract content via Document Intelligence
    doc_client = DocumentAnalysisClient(
        endpoint=os.environ['DOCINT_ENDPOINT'],
        credential=os.environ['DOCINT_KEY']
    )
    result = doc_client.begin_analyze_document('prebuilt-layout', blob).result()

    # 2. Chunk (400–800 tokens, 15% overlap)
    chunks = chunk_text(result.content, chunk_size=600, overlap=0.15)

    # 3. VERSION CONTROL — delete old chunks before inserting new ones
    search_client = SearchClient(
        endpoint=os.environ['SEARCH_ENDPOINT'],
        index_name='windturbine-index',
        credential=os.environ['SEARCH_KEY']
    )
    doc_id  = hashlib.md5(blob.name.encode()).hexdigest()
    old_docs = list(search_client.search('*', filter=f"document_id eq '{doc_id}'"))
    if old_docs:
        search_client.delete_documents([{'Id': d['Id']} for d in old_docs])
        logging.info(f'Deleted {len(old_docs)} stale chunks')

    # 4. Embed and index new chunks
    aoai = AzureOpenAI(api_key=os.environ['AOAI_API_KEY'],
                       azure_endpoint=os.environ['AOAI_ENDPOINT'],
                       api_version='2024-02-01')
    docs = []
    for i, chunk in enumerate(chunks):
        emb = aoai.embeddings.create(input=chunk, model='text-embedding-ada-002').data[0].embedding
        docs.append({
            'Id': f'{doc_id}-{i}',
            'Procedure_Chunk': chunk,
            'document_id': doc_id,
            'document_version': hashlib.md5(blob.read()).hexdigest()[:8],
            'version_updated_at': datetime.now(timezone.utc).isoformat(),
            'embedding': emb
        })
    search_client.upload_documents(docs)
    logging.info(f'Indexed {len(docs)} chunks for {blob.name}')

---
# Phase 5 — Monitoring & Observability
> Wire up Application Insights, set alerts for known failure modes, and track RAG-specific metrics.

## Step 5.1 — Create Log Analytics Workspace + Application Insights

### 🖥️ Azure Portal
**Log Analytics Workspace:**
1. Search **'Log Analytics workspaces'** → **+ Create**
2. Fill in:
   - **Resource group:** `rg-windturbine-prod`
   - **Name:** `law-windturbine`
   - **Region:** `East US`
3. Click **Review + create** → **Create**

**Application Insights:**
4. Search **'Application Insights'** → **+ Create**
5. Fill in:
   - **Resource group:** `rg-windturbine-prod`
   - **Name:** `appi-windturbine`
   - **Region:** `East US`
   - **Resource mode:** `Workspace-based`
   - **Log Analytics Workspace:** `law-windturbine`
6. Click **Review + create** → **Create**
7. After creation: go to resource → **Configure → Properties** → copy **Instrumentation Key**
   - Store in Key Vault: `APPINSIGHTS-KEY`

## Step 5.2 — Set Up Azure Monitor Alerts
Based on real production failure modes discovered during the project.

### 🖥️ Azure Portal
1. Go to **Azure Monitor** → **Alerts** → **+ Create → Alert rule**
2. **Scope:** select `appi-windturbine` (Application Insights resource)
3. **Condition:** click **Add condition**

**Alert 1 — High latency:**
- Signal: `Server response time` | Threshold: `Greater than 3000 ms` | Aggregation: `Average` over 10 min

**Alert 2 — Error rate spike:**
- Signal: `Failed requests` | Threshold: `Greater than 5` | Aggregation: `Count` over 5 min

**Alert 3 — Replica scaling spike:**
- Scope: select your Container App | Signal: `Replica count` | Threshold: `Greater than 8`

4. **Actions** tab: **+ Create action group** → add email/Teams webhook
5. **Details** tab: give the alert a name → **Review + create**

In [ ]:
# ⌨️ Azure CLI — create alerts
BACKEND_ID=$(az containerapp show --name $BACKEND_APP --resource-group $RESOURCE_GROUP --query id -o tsv)
APPI_ID=$(az monitor app-insights component show --app $APP_INSIGHTS --resource-group $RESOURCE_GROUP --query id -o tsv)

# Alert 1: High latency (>3s)
az monitor metrics alert create \
  --name 'alert-high-latency' --resource-group $RESOURCE_GROUP \
  --scopes $APPI_ID \
  --condition 'avg requests/duration > 3000' \
  --window-size 10m --evaluation-frequency 2m \
  --description 'Avg response > 3s — check Cosmos DB index and Redis'

# Alert 2: OpenAI 429 errors
az monitor metrics alert create \
  --name 'alert-openai-429' --resource-group $RESOURCE_GROUP \
  --scopes $APPI_ID \
  --condition 'count requests where resultCode == 429 > 5' \
  --window-size 5m --evaluation-frequency 1m \
  --description 'OpenAI rate limit — check Redis cache and request queue'

# Alert 3: Container scaling spike
az monitor metrics alert create \
  --name 'alert-scaling-spike' --resource-group $RESOURCE_GROUP \
  --scopes $BACKEND_ID \
  --condition 'avg Replicas > 8' \
  --window-size 5m --evaluation-frequency 1m \
  --description 'Replica count > 8 — possible traffic spike'

echo 'Monitor alerts configured'

## Step 5.3 — Instrument FastAPI with Custom Trace Spans

In [ ]:
# pip install opencensus-ext-azure
from opencensus.ext.azure.trace_exporter import AzureExporter
from opencensus.trace.tracer import Tracer
from opencensus.trace.samplers import AlwaysOnSampler
import os

def get_tracer():
    return Tracer(
        exporter=AzureExporter(
            connection_string=f"InstrumentationKey={os.environ['APPINSIGHTS_KEY']}"
        ),
        sampler=AlwaysOnSampler()
    )

# Example usage in FastAPI query endpoint
async def query(request):
    tracer = get_tracer()

    with tracer.span(name='redis_cache_lookup') as span:
        cached = await redis.get(request.question)
        span.add_attribute('cache_hit', cached is not None)
        if cached: return cached

    with tracer.span(name='azure_ai_search') as span:
        results = await search(request.question, top_k=request.top_k)
        span.add_attribute('chunks_returned', len(results))
        span.add_attribute('top_reranker_score', results[0].reranker_score if results else 0)

    with tracer.span(name='openai_generation') as span:
        response = await generate(request.question, results)
        span.add_attribute('prompt_tokens', response.usage.prompt_tokens)
        span.add_attribute('completion_tokens', response.usage.completion_tokens)

## Step 5.4 — KQL Queries for RAG Pipeline Health
Run these in **Azure Portal → Monitor → Log Analytics → Logs**.

### 🖥️ Azure Portal
1. Go to **Azure Monitor** → **Logs**
2. Select scope: `law-windturbine`
3. Paste any query below → click **Run**
4. To save as dashboard: click **Pin to dashboard** → select or create a shared dashboard

In [ ]:
# KQL 1 — Average chunks retrieved per hour (target: ≥ 3)
kql_chunks = """
customEvents
| where name == 'azure_ai_search'
| extend chunks = toint(customDimensions.chunks_returned)
| summarize avg_chunks = avg(chunks), min_chunks = min(chunks) by bin(timestamp, 1h)
| render timechart
"""

# KQL 2 — OpenAI token usage per day
kql_tokens = """
customEvents
| where name == 'openai_generation'
| extend total = toint(customDimensions.prompt_tokens) + toint(customDimensions.completion_tokens)
| summarize daily_tokens = sum(total) by bin(timestamp, 1d)
| render columnchart
"""

# KQL 3 — Redis cache hit rate
kql_cache = """
customEvents
| where name == 'redis_cache_lookup'
| extend hit = tobool(customDimensions.cache_hit)
| summarize hits = countif(hit == true), misses = countif(hit == false) by bin(timestamp, 1h)
| extend hit_rate = round(100.0 * hits / (hits + misses), 1)
| render timechart
"""

# KQL 4 — Fallback rate (< 3 chunks returned)
kql_fallback = """
customEvents
| where name == 'azure_ai_search'
| extend chunks = toint(customDimensions.chunks_returned)
| summarize total = count(), fallback = countif(chunks < 3) by bin(timestamp, 1h)
| extend fallback_rate = round(100.0 * fallback / total, 1)
| render timechart
"""

print('Paste any of the above KQL queries into Azure Monitor → Log Analytics → Logs')